<a href="https://colab.research.google.com/github/atsushi729/NLP-playbook/blob/main/Skipgram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# corpus
corpus = "I like natural language processing and machine learning".lower().split()

# vocabulary
vocab = list(set(corpus))
word_to_idx = {w:i for i,w in enumerate(vocab)}
idx_to_word = {i:w for w,i in word_to_idx.items()}
vocab_size = len(vocab)

# parameters
window_size = 2
embedding_dim = 20
negative_samples = 5

# training pairs
pairs = []
for i,word in enumerate(corpus):
    target = word_to_idx[word]
    for j in range(-window_size,window_size+1):
        if j==0: continue
        if 0 <= i+j < len(corpus):
            context = word_to_idx[corpus[i+j]]
            pairs.append((target,context))

# model
class SGNS(nn.Module):
    def __init__(self,vocab_size,embed_dim):
        super().__init__()
        self.input_embed = nn.Embedding(vocab_size,embed_dim)
        self.output_embed = nn.Embedding(vocab_size,embed_dim)

    def forward(self,target,context,negatives):
        v = self.input_embed(target)
        u = self.output_embed(context)
        score = torch.sum(v*u,dim=1)
        pos_loss = torch.log(torch.sigmoid(score))

        neg_u = self.output_embed(negatives)
        neg_score = torch.bmm(neg_u, v.unsqueeze(2)).squeeze(-1) # Changed .squeeze() to .squeeze(-1)
        neg_loss = torch.sum(torch.log(torch.sigmoid(-neg_score)),dim=1)

        return -(pos_loss + neg_loss).mean()

model = SGNS(vocab_size,embedding_dim)
optimizer = optim.Adam(model.parameters(),lr=0.01)

# training
for epoch in range(200):

    total_loss = 0

    for target,context in pairs:

        negatives = torch.tensor(
            random.sample(range(vocab_size),negative_samples)
        )

        target = torch.tensor([target])
        context = torch.tensor([context])
        negatives = negatives.unsqueeze(0)

        optimizer.zero_grad()

        loss = model(target,context,negatives)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 50 == 0:
        print(epoch,total_loss)

0 276.8149703145027
50 54.58625936508179
100 52.93895876407623
150 50.09678888320923


In [4]:
import torch.nn.functional as F

def get_similarity(word1, word2):
    if word1 not in word_to_idx or word2 not in word_to_idx:
        return "Word not in vocab"

    # 入力側の埋め込みベクトルを取得
    v1 = model.input_embed(torch.tensor([word_to_idx[word1]]))
    v2 = model.input_embed(torch.tensor([word_to_idx[word2]]))

    # コサイン類似度を計算 (1に近いほど似ている)
    sim = F.cosine_similarity(v1, v2)
    return sim.item()

# 例: 'learning' と他の単語の類似度を比較
test_word = 'learning'
print(f"--- Similarity with '{test_word}' ---")
for w in vocab:
    if w != test_word:
        print(f"{w}: {get_similarity(test_word, w):.4f}")

--- Similarity with 'learning' ---
machine: 0.0122
i: 0.0057
and: 0.3857
processing: 0.0079
language: 0.1046
like: 0.0958
natural: -0.1573


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np

def visualize_embeddings(model, vocab, n_words=100):
    # 最初のn個の単語のベクトルを取り出す
    embeddings = model.input_embed.weight.detach().cpu().numpy()[:n_words]
    words = vocab[:n_words]

    # t-SNEで2次元に圧縮
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(words)-1))
    vectors_2d = tsne.fit_transform(embeddings)

    # 描画
    plt.figure(figsize=(12, 12))
    for i, word in enumerate(words):
        plt.scatter(vectors_2d[i, 0], vectors_2d[i, 1])
        plt.annotate(word, (vectors_2d[i, 0], vectors_2d[i, 1]), alpha=0.7)
    plt.title("Word Embedding Visualization (t-SNE)")
    plt.grid(True)
    plt.show()

visualize_embeddings(model, vocab)

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

def visualize_embeddings(model, vocab, n_words=100):
    # 埋め込み層の重みを取得
    embeddings = model.input_embed.weight.detach().cpu().numpy()[:n_words]
    labels = vocab[:n_words]

    # t-SNEで2次元に圧縮
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, n_words-1))
    vectors_2d = tsne.fit_transform(embeddings)

    # プロット
    plt.figure(figsize=(12, 10))
    plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], edgecolors='k', c='lightblue')

    for i, label in enumerate(labels):
        plt.annotate(label, (vectors_2d[i, 0], vectors_2d[i, 1]), xytext=(5, 2),
                     textcoords='offset points', ha='right', va='bottom', fontsize=9)

    plt.title("t-SNE Visualization of Word Embeddings (WikiText-2 Subset)")
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

visualize_embeddings(model, vocab, n_words=150)

In [ ]:
# 'he' という単語に割り当てられている 50 次元のベクトルの「生データ」を見てみます
if 'he' in word_to_idx:
    he_idx = torch.tensor([word_to_idx['he']])
    # grad_fn が付いているのは、これが「学習によって更新される値」であることを示しています
    he_vector = model.input_embed(he_idx)

    print(f"単語 'he' のベクトル形式 (最初の10次元のみ):")
    print(he_vector.detach().numpy()[0][:10])

    print(f"\nベクトルの形状: {he_vector.shape} (embedding_dim が 50 なので 50 個の数値があります)")

In [5]:
!pip install datasets

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import random
from collections import Counter
from datasets import load_dataset
import re

# 1. 大規模データのロード (WikiText-2のサブセットを使用)
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
# 最初の1000行程度に制限して学習時間を調整
raw_text = " ".join(dataset['text'][:1000]).lower()
words = re.findall(r'\w+', raw_text)

# 2. 語彙の構築 (頻度の低い単語を除外)
word_counts = Counter(words)
vocab = [w for w, c in word_counts.items() if c > 5]
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")

# 3. データセットクラスの定義
class SkipGramDataset(Dataset):
    def __init__(self, words, word_to_idx, window_size=2):
        self.data = []
        indices = [word_to_idx[w] for w in words if w in word_to_idx]
        for i, target in enumerate(indices):
            for j in range(max(0, i - window_size), min(len(indices), i + window_size + 1)):
                if i != j:
                    self.data.append((target, indices[j]))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

# 4. モデルの再定義 (バッチ処理対応)
class SGNS_Large(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.input_embed = nn.Embedding(vocab_size, embed_dim)
        self.output_embed = nn.Embedding(vocab_size, embed_dim)

    def forward(self, target, context, negatives):
        # target: [batch_size], context: [batch_size], negatives: [batch_size, neg_samples]
        v = self.input_embed(target) # [batch_size, embed_dim]
        u = self.output_embed(context) # [batch_size, embed_dim]

        pos_score = torch.sum(v * u, dim=1) # [batch_size]
        pos_loss = torch.log(torch.sigmoid(pos_score))

        neg_u = self.output_embed(negatives) # [batch_size, neg_samples, embed_dim]
        neg_score = torch.bmm(neg_u, v.unsqueeze(2)).squeeze(-1) # [batch_size, neg_samples]
        neg_loss = torch.sum(torch.log(torch.sigmoid(-neg_score)), dim=1)

        return -(pos_loss + neg_loss).mean()

# 5. 訓練の実行
window_size = 2
embedding_dim = 50
negative_samples = 5
batch_size = 512

ds = SkipGramDataset(words, word_to_idx, window_size)
dl = DataLoader(ds, batch_size=batch_size, shuffle=True)

model = SGNS_Large(vocab_size, embedding_dim)
optimizer = optim.Adam(model.parameters(), lr=0.005)

print("Training...")
for epoch in range(10):
    total_loss = 0
    for target, context in dl:
        negatives = torch.randint(0, vocab_size, (target.size(0), negative_samples))

        optimizer.zero_grad()
        loss = model(target, context, negatives)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dl):.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Vocab size: 1233
Training...
Epoch 1, Loss: 12.2759
Epoch 2, Loss: 6.1368
Epoch 3, Loss: 3.5321
Epoch 4, Loss: 2.6349
Epoch 5, Loss: 2.2669
Epoch 6, Loss: 2.0732
Epoch 7, Loss: 1.9408
Epoch 8, Loss: 1.8448
Epoch 9, Loss: 1.7711
Epoch 10, Loss: 1.7102


In [7]:
def get_similarity_updated(word1, word2):
    if word1 not in word_to_idx or word2 not in word_to_idx:
        return "Word not in vocab"
    v1 = model.input_embed(torch.tensor([word_to_idx[word1]]))
    v2 = model.input_embed(torch.tensor([word_to_idx[word2]]))
    return torch.nn.functional.cosine_similarity(v1, v2).item()

test_word = 'he'
if test_word in word_to_idx:
    print(f"--- Similarity with '{test_word}' ---")
    results = []
    for w in vocab[:100]: # 最初の100語から比較
        if w != test_word:
            results.append((w, get_similarity_updated(test_word, w)))
    for w, s in sorted(results, key=lambda x: x[1], reverse=True)[:10]:
        print(f"{w}: {s:.4f}")
else:
    print(f"'{test_word}' is not in vocab.")

--- Similarity with 'he' ---
that: 0.6950
it: 0.6627
was: 0.6621
an: 0.6574
during: 0.6246
a: 0.5607
to: 0.5307
time: 0.5279
for: 0.5184
by: 0.5098
